# PyTorch Custom TokenDataset

Implementation of a custom PyTorch Dataset for token sequences.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
class TokenDataset(Dataset):
    """Custom Dataset for token sequences with automatic padding."""
    
    def __init__(self, data, max_length=None):
        self.data = data
        if max_length is None:
            self.max_length = max(len(item['tokens']) for item in data)
        else:
            self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        tokens = item['tokens']
        label = item['label']
        
        # Pad tokens to max_length
        padded = tokens + [0] * (self.max_length - len(tokens))
        
        return torch.tensor(padded, dtype=torch.long), torch.tensor(label, dtype=torch.long)

## Example Usage

This demonstrates how to use the TokenDataset with sample data.

In [ ]:
# Sample data
sample_data = [
    {'tokens': [1, 2, 3], 'label': 0},
    {'tokens': [4, 5], 'label': 1},
    {'tokens': [6, 7, 8, 9], 'label': 0}
]

# Create dataset
dataset = TokenDataset(sample_data)
print(f"Dataset size: {len(dataset)}")
print(f"Max sequence length: {dataset.max_length}")

## CSVDataset Implementation

Extending support to CSV data format.

In [ ]:
class CSVDataset(Dataset):
    """Custom Dataset for CSV data with one-hot encoding."""
    
    def __init__(self, csv_path):
        self.data = pd.read_csv(csv_path)
        self.features = self.data.iloc[:, :-1].values
        self.labels = self.data.iloc[:, -1].values
        
        # One-hot encode labels
        self.num_classes = len(np.unique(self.labels))
        self.labels_encoded = np.eye(self.num_classes)[self.labels]
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        features = torch.tensor(self.features[idx], dtype=torch.float32)
        label = torch.tensor(self.labels_encoded[idx], dtype=torch.float32)
        return features, label